In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StructType, StructField, StringType
from pyspark.sql.session import SparkSession

spark = SparkSession.getActiveSession()

In [0]:
dbutils.widgets.text("catalog", "dsit_mcp", "Catalog")
dbutils.widgets.text("bronze_schema", "01_bronze", "Bronze Schema")
dbutils.widgets.text("silver_schema", "02_silver", "Silver Schema")

catalog = dbutils.widgets.get("catalog")
bronze_schema = dbutils.widgets.get("bronze_schema")
silver_schema = dbutils.widgets.get("silver_schema")

bronze_table = "bronze_pupil_dest_data"

In [0]:
bronze_df = spark.table(f"`{catalog}`.`{bronze_schema}`.`{bronze_table}`")


In [0]:
non_null_cols = [c for c in bronze_df.columns if bronze_df.filter(F.col(c).isNotNull()).limit(1).count() > 0]
bronze_df_no_all_nulls = bronze_df.select(non_null_cols)

In [0]:
display(bronze_df_no_all_nulls)

In [0]:
# Rename columns after 'education' with descriptive UK pupil destination names
column_renames = {
    "he": "higher_education",
    "fe": "further_education",
    "fel3": "further_ed_level_3",
    "fel2": "further_ed_level_2",
    "fel1": "further_ed_level_1",
    "other_edu": "other_education",
    "appren": "apprenticeships",
    "appl4": "apprenticeship_level_4_plus",
    "appl3": "apprenticeship_level_3",
    "appl2": "apprenticeship_level_2",
    "all_work": "employment",
    "all_notsust": "not_sustained_destination",
    "all_unknown": "destination_unknown"
}

# Apply the renames
for old_name, new_name in column_renames.items():
    bronze_df_no_all_nulls = bronze_df_no_all_nulls.withColumnRenamed(old_name, new_name)

display(bronze_df_no_all_nulls)

In [0]:
# Generate CREATE TABLE statement with column comments for silver_pupil_dest_data
create_table_sql = f"""
CREATE TABLE IF NOT EXISTS `{catalog}`.`{silver_schema}`.`silver_pupil_dest_data` (
  time_period INT COMMENT 'Academic year in format YYYYYY (e.g., 201617 for 2016/17)',
  time_identifier STRING COMMENT 'Time period type (e.g., Academic year)',
  geographic_level STRING COMMENT 'Geographic aggregation level (e.g., Local authority, Regional, National)',
  country_code STRING COMMENT 'ONS country code',
  country_name STRING COMMENT 'Country name (e.g., England)',
  region_code STRING COMMENT 'ONS region code',
  region_name STRING COMMENT 'Region name (e.g., North East, London)',
  old_la_code INT COMMENT 'Legacy local authority code',
  new_la_code STRING COMMENT 'Current local authority code',
  la_name STRING COMMENT 'Local authority name',
  pcon_code STRING COMMENT 'Parliamentary constituency code',
  pcon_name STRING COMMENT 'Parliamentary constituency name',
  lad_code STRING COMMENT 'Local authority district code',
  lad_name STRING COMMENT 'Local authority district name',
  institution_group STRING COMMENT 'Type of institution group (e.g., State-funded mainstream schools)',
  institution_type STRING COMMENT 'Specific institution type classification',
  cohort_level_group STRING COMMENT 'Qualification level grouping',
  cohort_level STRING COMMENT 'Specific qualification level achieved',
  breakdown_topic STRING COMMENT 'Demographic breakdown category (e.g., Disadvantage Status, Ethnicity, Sex)',
  breakdown STRING COMMENT 'Specific demographic group within the breakdown topic',
  data_type STRING COMMENT 'Type of metric (Number of students or Percentage)',
  version STRING COMMENT 'Data version status (e.g., Revised, Final)',
  cohort STRING COMMENT 'Total number of students in the cohort',
  overall STRING COMMENT 'Overall destination rate or count',
  education STRING COMMENT 'Students continuing in any form of education',
  higher_education STRING COMMENT 'Students progressing to higher education (universities)',
  further_education STRING COMMENT 'Students in further education (all levels)',
  further_ed_level_3 STRING COMMENT 'Students in further education at Level 3 (A-level equivalent)',
  further_ed_level_2 STRING COMMENT 'Students in further education at Level 2 (GCSE equivalent)',
  further_ed_level_1 STRING COMMENT 'Students in further education at Level 1 (foundation level)',
  other_education STRING COMMENT 'Students in other education not classified above',
  apprenticeships STRING COMMENT 'Students in apprenticeships (all levels)',
  apprenticeship_level_4_plus STRING COMMENT 'Students in Level 4+ apprenticeships (degree apprenticeships)',
  apprenticeship_level_3 STRING COMMENT 'Students in Level 3 apprenticeships (advanced)',
  apprenticeship_level_2 STRING COMMENT 'Students in Level 2 apprenticeships (intermediate)',
  employment STRING COMMENT 'Students in sustained employment without education',
  not_sustained_destination STRING COMMENT 'Students without sustained education or employment',
  destination_unknown STRING COMMENT 'Students with unknown destination status'
)
COMMENT 'This release provides information on destinations from:

state-funded mainstream schools
state-funded special schools (including non-maintained special schools)
alternative provision
This information is based on data from the National Pupil Database and the Longitudinal Education Outcomes dataset.

The following symbols have been used in this publication: 

( 0 ) zero 

( c ) small number suppressed to preserve confidentiality or for accountability reasons 

( z ) not applicable 

(x) not available

( low ) positive % less than 0.5'
"""

In [0]:
spark.sql(create_table_sql)

In [0]:
# Select only columns matching the target table schema
silver_columns = [
    'time_period', 'time_identifier', 'geographic_level', 'country_code', 'country_name', 'region_code', 'region_name',
    'old_la_code', 'new_la_code', 'la_name', 'pcon_code', 'pcon_name', 'lad_code', 'lad_name', 'institution_group',
    'institution_type', 'cohort_level_group', 'cohort_level', 'breakdown_topic', 'breakdown', 'data_type', 'version',
    'cohort', 'overall', 'education', 'higher_education', 'further_education', 'further_ed_level_3', 'further_ed_level_2',
    'further_ed_level_1', 'other_education', 'apprenticeships', 'apprenticeship_level_4_plus', 'apprenticeship_level_3',
    'apprenticeship_level_2', 'employment', 'not_sustained_destination', 'destination_unknown'
]
bronze_df_to_write = bronze_df_no_all_nulls.select([c for c in silver_columns if c in bronze_df_no_all_nulls.columns])
bronze_df_to_write.write.mode("append").saveAsTable(f"`{catalog}`.`{silver_schema}`.`silver_pupil_dest_data`")